In [26]:
team_matches.to_csv("../data/features/match_features.csv", index=False)
team_matches.shape

(2280, 23)

In [25]:
import os
os.makedirs("../data/features", exist_ok=True)

In [24]:
team_matches = team_matches.merge(
    h2h_df[["team", "utcDate", "h2h_win_pct"]],
    on=["team", "utcDate"],
    how="left"
)

In [23]:
h2h_df[h2h_df["h2h_win_pct"].notna()].head(10)

,team,opponent,utcDate,goals_for,goals_against,h2h_win,h2h_win_pct
1,AFC Bournemouth,Arsenal FC,2024-05-04 11:30:00+00:00,0,3,0,0.000000
2,AFC Bournemouth,Arsenal FC,2024-10-19 16:30:00+00:00,2,0,1,0.000000
3,AFC Bournemouth,Arsenal FC,2025-05-03 16:30:00+00:00,2,1,1,0.333333
4,AFC Bournemouth,Arsenal FC,2026-01-03 17:30:00+00:00,2,3,0,0.500000
5,AFC Bournemouth,Arsenal FC,2026-04-11 11:30:00+00:00,2,1,1,0.400000
7,AFC Bournemouth,Aston Villa FC,2024-04-21 14:00:00+00:00,1,3,0,0.000000
8,AFC Bournemouth,Aston Villa FC,2024-10-26 14:00:00+00:00,1,1,0,0.000000
9,AFC Bournemouth,Aston Villa FC,2025-05-10 16:30:00+00:00,0,1,0,0.000000
10,AFC Bournemouth,Aston Villa FC,2025-11-09 14:00:00+00:00,0,4,0,0.000000
11,AFC Bournemouth,Aston Villa FC,2026-02-07 15:00:00+00:00,1,1,0,0.000000


In [21]:
h2h_df["h2h_win"] = (h2h_df["goals_for"] > h2h_df["goals_against"]).astype(int)

h2h_df["h2h_win_pct"] = (
    h2h_df.groupby(["team", "opponent"])["h2h_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [20]:
h2h_records = []

for _, match in clean_matches.iterrows():
    h2h_records.append({"team": match["home_team_name"], "opponent": match["away_team_name"],
                         "utcDate": match["utcDate"], "goals_for": match["home_goals"], "goals_against": match["away_goals"]})
    h2h_records.append({"team": match["away_team_name"], "opponent": match["home_team_name"],
                         "utcDate": match["utcDate"], "goals_for": match["away_goals"], "goals_against": match["home_goals"]})

h2h_df = pd.DataFrame(h2h_records).sort_values(["team", "opponent", "utcDate"]).reset_index(drop=True)

In [19]:
team_matches[team_matches["team"] == "AFC Bournemouth"][
    ["utcDate", "is_home", "goal_difference", "clean_sheet_pct", "days_since_last_match", "home_win_pct", "away_win_pct"]
].head(10)

,utcDate,is_home,goal_difference,clean_sheet_pct,days_since_last_match,home_win_pct,away_win_pct
0,2023-08-12 14:00:00+00:00,True,0,NaN,NaN,NaN,NaN
1,2023-08-19 14:00:00+00:00,False,-2,0.000000,7.0,NaN,NaN
2,2023-08-26 11:30:00+00:00,True,-2,0.000000,6.0,0.0,NaN
3,2023-09-02 14:00:00+00:00,False,0,0.000000,7.0,NaN,0.0
4,2023-09-17 13:00:00+00:00,True,0,0.000000,14.0,0.0,NaN
5,2023-09-24 13:00:00+00:00,False,-2,0.200000,7.0,NaN,0.0
6,2023-09-30 14:00:00+00:00,True,-4,0.166667,6.0,0.0,NaN
7,2023-10-07 14:00:00+00:00,False,-3,0.142857,7.0,NaN,0.0
8,2023-10-21 14:00:00+00:00,True,-1,0.125000,14.0,0.0,NaN
9,2023-10-28 14:00:00+00:00,True,1,0.111111,7.0,0.0,NaN


In [18]:
team_matches["is_win"] = (team_matches["result"] == "WIN").astype(int)

team_matches["home_win_pct"] = (
    team_matches[team_matches["is_home"]]
    .groupby(["team", "season"])["is_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_matches["away_win_pct"] = (
    team_matches[~team_matches["is_home"]]
    .groupby(["team", "season"])["is_win"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [16]:
team_matches["days_since_last_match"] = (
    team_matches.groupby(["team", "season"])["utcDate"]
    .diff()
    .dt.days
)

In [15]:
team_matches["clean_sheet"] = (team_matches["goals_conceded"] == 0).astype(int)
team_matches["clean_sheet_pct"] = (
    team_matches.groupby(["team", "season"])["clean_sheet"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [14]:
team_matches["goal_difference"] = team_matches["goals_scored"] - team_matches["goals_conceded"]

In [13]:
team_matches[team_matches["team"] == "AFC Bournemouth"][
    ["utcDate", "result", "points", "rolling_form_pts", "win_streak", "loss_streak", "avg_goals_scored", "avg_goals_conceded"]
].head(10)

,utcDate,result,points,rolling_form_pts,win_streak,loss_streak,avg_goals_scored,avg_goals_conceded
0,2023-08-12 14:00:00+00:00,DRAW,1,NaN,0,0,NaN,NaN
1,2023-08-19 14:00:00+00:00,LOSS,0,1.0,0,0,1.000000,1.000000
2,2023-08-26 11:30:00+00:00,LOSS,0,1.0,0,1,1.000000,2.000000
3,2023-09-02 14:00:00+00:00,DRAW,1,1.0,0,2,0.666667,2.000000
4,2023-09-17 13:00:00+00:00,DRAW,1,2.0,0,0,1.000000,2.000000
5,2023-09-24 13:00:00+00:00,LOSS,0,3.0,0,0,0.800000,1.600000
6,2023-09-30 14:00:00+00:00,LOSS,0,2.0,0,1,0.833333,1.833333
7,2023-10-07 14:00:00+00:00,LOSS,0,2.0,0,2,0.714286,2.142857
8,2023-10-21 14:00:00+00:00,LOSS,0,2.0,0,3,0.625000,2.250000
9,2023-10-28 14:00:00+00:00,WIN,3,1.0,0,4,0.666667,2.222222


In [12]:
team_matches["avg_goals_scored"] = (
    team_matches.groupby(["team", "season"])["goals_scored"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
team_matches["avg_goals_conceded"] = (
    team_matches.groupby(["team", "season"])["goals_conceded"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [11]:
def compute_streak(results, target):
    streak = 0
    streaks = []
    for r in results:
        streaks.append(streak)  # streak BEFORE this match
        streak = streak + 1 if r == target else 0
    return streaks

team_matches["win_streak"] = team_matches.groupby(["team", "season"])["result"].transform(
    lambda x: compute_streak(x, "WIN")
)
team_matches["loss_streak"] = team_matches.groupby(["team", "season"])["result"].transform(
    lambda x: compute_streak(x, "LOSS")
)

In [10]:
team_matches["rolling_form_pts"] = (
    team_matches.groupby(["team", "season"])["points"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

In [9]:
points_map = {"WIN": 3, "DRAW": 1, "LOSS": 0}
team_matches["points"] = team_matches["result"].map(points_map)

In [8]:
home_rows = clean_matches[["utcDate", "matchday", "season", "id", "home_team_name", "home_goals", "away_goals", "winner"]].copy()
home_rows.columns = ["utcDate", "matchday", "season", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
home_rows["is_home"] = True
home_rows["result"] = home_rows["winner"].map({"HOME_TEAM": "WIN", "AWAY_TEAM": "LOSS", "DRAW": "DRAW"})

away_rows = clean_matches[["utcDate", "matchday", "season", "id", "away_team_name", "away_goals", "home_goals", "winner"]].copy()
away_rows.columns = ["utcDate", "matchday", "season", "match_id", "team", "goals_scored", "goals_conceded", "winner"]
away_rows["is_home"] = False
away_rows["result"] = away_rows["winner"].map({"HOME_TEAM": "LOSS", "AWAY_TEAM": "WIN", "DRAW": "DRAW"})

team_matches = pd.concat([home_rows, away_rows]).sort_values(["team", "utcDate"]).reset_index(drop=True)
team_matches.drop(columns="winner", inplace=True)

In [7]:
import pandas as pd

clean_matches = pd.read_csv("../data/processed/clean_matches_multi_season.csv", parse_dates=["utcDate"])
clean_matches = clean_matches.sort_values("utcDate").reset_index(drop=True)